# Stage 2 — Validation

(i) DCM null (50 draws), (ii) seeded Louvain/Infomap comparison, (iii) five-schedule reseed campaign with matched-control ablations and the dark-matter stream, (iv) self-containment via the consensus table; then the arXiv label check. Long stages (hours) sit behind `RUN_LONG`; their products of record ship in `data/communities/` and `data/communities/campaign/` (the campaign ledger).

In [ ]:
import os, pathlib, sys
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "scripts").is_dir() and (root / "data").is_dir(): break
    root = root.parent
else:
    raise SystemExit("repository root (containing scripts/ and data/) not found within 6 levels")
os.chdir(root); print("working directory:", os.getcwd())
assert "igraph" in {m.split("==")[0] for m in os.popen(f"{sys.executable} -m pip list --format=freeze 2>/dev/null").read().split()}, \
    f"kernel {sys.executable} lacks python-igraph: select the grb-venv kernel (see notebooks/README.md)"

In [ ]:
CORPUS = "data/raw/ads_corpus_v2_core_frozen.jsonl"  # local-only frozen corpus (ADS terms); see README
import pathlib
HAVE_CORPUS = pathlib.Path(CORPUS).exists()
print("frozen corpus present:", HAVE_CORPUS)
RUN_LONG = False
RUN_FETCH = False

(i) DCM null: fifty draws at ΔT = 1 yr and the ΔT = 2 yr check (long; `--realisations 50` is required — the script default is 20).

In [ ]:
if RUN_LONG and HAVE_CORPUS:
    %run scripts/dcm_null.py --layer-years 1 --realisations 50
    %run scripts/dcm_null.py --layer-years 2 --realisations 50
else:
    print("skipped; products of record: dcm_null_dT1.json / dcm_null_dT2.json")

(ii) Cross-algorithm agreement on the frozen graph, seeded.

In [ ]:
if HAVE_CORPUS:
    %run scripts/algorithm_comparison2.py
else:
    print("skipped; product of record: algorithm_comparison2.json")

(iii-a) Representation checks.

In [ ]:
if HAVE_CORPUS:
    %run scripts/representation_check.py
if RUN_LONG and HAVE_CORPUS:
    %run scripts/representation_three.py
    %run scripts/canonical_directed.py
else:
    print("consensus-level representation products are in data/communities/")

(iii-b) The five-schedule reseed campaign (hours) and the base ablation-control pass that the repository-only pipeline diagram reads; the aggregate is cheap and always runs on the saved ledger.

In [ ]:
if RUN_LONG and HAVE_CORPUS:
    for j in range(5):
        %run scripts/campaign_reseed.py --j {j}
    %run scripts/ablation_controls2.py
%run scripts/campaign_aggregate.py

arXiv primary-category check on the labels (the metadata file ships with the repository; refetch only to rebuild it).

In [ ]:
if RUN_FETCH:
    %run scripts/fetch_arxiv_meta.py
%run scripts/validate_labels_arxiv.py